# Melbourne House Price Prediction (Regression)

Goal: predict the sale price of a house from its features.

Steps: load data, check and fix missing values, build a Pipeline, compare two models
(with and without SelectKBest), evaluate, save the pickle file.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load the data

In [ ]:
# change the path if your csv is in another folder
df = pd.read_csv('../data/Melbourne_housing.csv', low_memory=False)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Missing values

First look at how many values are missing in every column.

In [ ]:
missing = df.isnull().sum()
missing_percent = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing': missing, 'percent': missing_percent}).sort_values('percent', ascending=False)

The table does not show everything. `BuildingArea` is read as text, so some missing values
are hidden as words (like `missing`) or as `inf`. Those must be changed to real `NaN` first.

In [ ]:
# 'missing' becomes NaN when we convert the column to numbers
df['BuildingArea'] = pd.to_numeric(df['BuildingArea'], errors='coerce')
# inf is not a real area, so it also becomes NaN
df['BuildingArea'] = df['BuildingArea'].replace([np.inf, -np.inf], np.nan)
df['BuildingArea'].dtype

## 3. Choose the columns

Features: Rooms, Type, Distance, Bedroom, Bathroom, Car, Landsize, BuildingArea, YearBuilt, Regionname.
Target: Price.

In [ ]:
feature_cols = ['Rooms', 'Type', 'Distance', 'Bedroom', 'Bathroom', 'Car',
                'Landsize', 'BuildingArea', 'YearBuilt', 'Regionname']

data = df[feature_cols + ['Price']].copy()
data.shape

### Missing values in the target (Price)

We cannot train or test a model on rows where the answer is unknown, and filling the target
with a guess would be making up the answer. So these rows are removed.

In [ ]:
print('rows before:', len(data))
data = data.dropna(subset=['Price'])
print('rows after :', len(data))

### Impossible values

Some numbers cannot be real (a house built in year 2106, 30 bedrooms, a building area of 44515 m2).
Instead of deleting the whole row, I change the value to `NaN` and let the imputer fill it later.

In [ ]:
data.describe().T[['min', 'max']]

In [ ]:
data.loc[(data['YearBuilt'] < 1800) | (data['YearBuilt'] > 2026), 'YearBuilt'] = np.nan
data.loc[(data['BuildingArea'] <= 0) | (data['BuildingArea'] > 1000), 'BuildingArea'] = np.nan
data.loc[data['Landsize'] > 50000, 'Landsize'] = np.nan
data.loc[data['Bedroom'] > 10, 'Bedroom'] = np.nan
data.loc[data['Bathroom'] > 8, 'Bathroom'] = np.nan
data.loc[data['Car'] > 10, 'Car'] = np.nan

# Rooms has no missing values, so the few rows with more than 10 rooms are simply removed
data = data[data['Rooms'] <= 10]
data.shape

### Missing values left in the features

Why not use `dropna()`? Let's see how many rows we would lose.

In [ ]:
print('rows now           :', len(data))
print('rows after dropna():', len(data.dropna()))

In [ ]:
(data.isnull().sum() / len(data) * 100).round(1).sort_values(ascending=False)

Dropping every row with a missing value would throw away most of the data, so I fill the
missing values instead (like in Lesson 5 with `SimpleImputer`):

| Column type | Method | Why |
|---|---|---|
| Numbers (Bedroom, Bathroom, Car, Landsize, BuildingArea, YearBuilt, Distance) | `SimpleImputer(strategy='median')` | The median is not pulled by very large values (for example a huge Landsize), the mean is |
| Text columns (Type, Regionname) | `SimpleImputer(strategy='most_frequent')` | The most common category is the only sensible fill for text. They have no missing values now, but the app/pipeline stays safe if one appears |

The imputer is placed **inside the Pipeline**, so it learns the median only from the training
data and the same fill values are used again when the app makes a prediction.

## 4. Target: log of Price

Prices are skewed (a few very expensive houses). The log of the price is much more balanced,
and a linear model fits it better.

In [ ]:
print('skew of Price     :', round(data['Price'].skew(), 2))
print('skew of log(Price):', round(np.log(data['Price']).skew(), 2))

In [ ]:
X = data[feature_cols]
y = np.log(data['Price'])
X.shape, y.shape

## 5. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)
X_train.shape, X_test.shape

## 6. Preprocessing (ColumnTransformer)

In [ ]:
cat_cols = X.select_dtypes(exclude='number').columns   # Type, Regionname
num_cols = X.select_dtypes(include='number').columns   # the rest
print(list(cat_cols))
print(list(num_cols))

In [ ]:
# numbers: fill missing with median, then scale
num_steps = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# text: fill missing with most frequent, then one-hot encode
cat_steps = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessing = ColumnTransformer([
    ('numcolumns', num_steps, num_cols),
    ('catcolumns', cat_steps, cat_cols)
])
preprocessing

In [ ]:
# how many columns does the model really see after encoding?
preprocessing.fit_transform(X_train).shape[1]

## 7. Two models

* **pipeline1**: preprocessing + SelectKBest + LinearRegression (feature selection)
* **pipeline2**: preprocessing + LinearRegression (all columns)

Each model gets its own `LinearRegression()` so they do not share fitted values.

In [ ]:
def make_pipeline1(k):
    return Pipeline([
        ('preprocessing', preprocessing),
        ('selection', SelectKBest(f_regression, k=k)),
        ('modelling', LinearRegression())
    ])

pipeline2 = Pipeline([
    ('preprocessing', preprocessing),
    ('modelling', LinearRegression())
])

### Evaluation function

The model predicts log(price), so I use `np.exp()` to turn predictions back into dollars
before calculating MAE and RMSE. R2 is calculated on the log scale (the scale the model is trained on).

In [ ]:
def get_scores(model):
    scores = {}
    for name, X_part, y_part in [('train', X_train, y_train), ('test', X_test, y_test)]:
        pred = model.predict(X_part)
        scores[name + '_R2'] = r2_score(y_part, pred)
        scores[name + '_MAE'] = mean_absolute_error(np.exp(y_part), np.exp(pred))
        scores[name + '_MSE'] = mean_squared_error(np.exp(y_part), np.exp(pred))
        scores[name + '_RMSE'] = np.sqrt(scores[name + '_MSE'])
    return scores

### Model 2: all columns (no SelectKBest)

In [ ]:
pipeline2.fit(X_train, y_train)
p2 = get_scores(pipeline2)
for key, value in p2.items():
    print(f"{key:10s}: {value:,.3f}")

### Model 1: SelectKBest with different k

`k` is the number of columns kept after encoding. I try the values from the lesson (5, 6, 7) and some bigger ones.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []

for k in [5, 6, 7, 10, 12, 15, 17, 19]:
    model = make_pipeline1(k)
    model.fit(X_train, y_train)
    s = get_scores(model)
    cv = cross_val_score(make_pipeline1(k), X_train, y_train, cv=kfold, scoring='r2').mean()
    rows.append({'model': f'SelectKBest k={k}', 'CV_R2': cv, 'test_R2': s['test_R2'],
                 'test_MAE': s['test_MAE'], 'test_RMSE': s['test_RMSE']})

cv2 = cross_val_score(pipeline2, X_train, y_train, cv=kfold, scoring='r2').mean()
rows.append({'model': 'All columns (pipeline2)', 'CV_R2': cv2, 'test_R2': p2['test_R2'],
             'test_MAE': p2['test_MAE'], 'test_RMSE': p2['test_RMSE']})

results = pd.DataFrame(rows).round(3)
results

Reading the table: when the score keeps going up as k gets bigger, the columns that were removed
were still useful. Once the score stops changing (the last rows), the extra columns add nothing new.
Compare the best SelectKBest row with the "All columns" row and keep the better one. With few columns
(k = 5, 6, 7) the score is much lower, so the full set of columns is the better model here.

## 8. Other things I tried to improve the result

Each test uses the same 5-fold cross validation on the training data.

In [ ]:
def cv_r2(strategy='median', target=None):
    num_part = Pipeline([('imputer', SimpleImputer(strategy=strategy)), ('scaler', StandardScaler())])
    pre = ColumnTransformer([('numcolumns', num_part, num_cols), ('catcolumns', cat_steps, cat_cols)])
    model = Pipeline([('preprocessing', pre), ('modelling', LinearRegression())])
    y_used = y_train if target is None else target
    return cross_val_score(model, X_train, y_used, cv=kfold, scoring='r2').mean()

print('median imputer, log price :', round(cv_r2('median'), 3))
print('mean imputer,   log price :', round(cv_r2('mean'), 3))
print('median imputer, raw price :', round(cv_r2('median', np.exp(y_train)), 3))

Mean and median imputation give almost the same score. I keep the median because it is not affected by extreme values.

The last line uses the real price instead of the log price. Note that this R2 is on the dollar scale,
so it is a different scale from the other two rows. The log price still gives the more balanced errors.

## 9. Final model

I keep the model with the best test score from section 7. Change `final_model` if your table says another model is better.

In [ ]:
final_model = pipeline2
final_scores = get_scores(final_model)

print(f"Train R2   : {final_scores['train_R2']:.3f}")
print(f"Test  R2   : {final_scores['test_R2']:.3f}")
print(f"Test  MAE  : ${final_scores['test_MAE']:,.0f}")
print(f"Test  RMSE : ${final_scores['test_RMSE']:,.0f}")

### Try one house from the test data

In [ ]:
one_house = X_test.iloc[[0]]
one_house

In [ ]:
predicted_price = np.exp(final_model.predict(one_house))[0]
real_price = np.exp(y_test.iloc[0])
print(f"Predicted: ${predicted_price:,.0f}")
print(f"Real     : ${real_price:,.0f}")

### Make a prediction from new values (same way the Streamlit app does it)

In [ ]:
new_house = pd.DataFrame([{
    'Rooms': 3, 'Type': 'h', 'Distance': 10.0, 'Bedroom': 3, 'Bathroom': 2, 'Car': 1,
    'Landsize': 400, 'BuildingArea': 150, 'YearBuilt': 1990, 'Regionname': 'Southern Metropolitan'
}])
print(f"${np.exp(final_model.predict(new_house))[0]:,.0f}")

### A house with missing values also works

The imputer inside the pipeline fills them, so the model does not crash.

In [ ]:
house_with_gaps = new_house.copy()
house_with_gaps['BuildingArea'] = np.nan
house_with_gaps['YearBuilt'] = np.nan
print(f"${np.exp(final_model.predict(house_with_gaps))[0]:,.0f}")

## 10. Save the model and the cleaned data

In [ ]:
# the Streamlit app expects these two files (paths are from the notebook folder)
with open('../house_price_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

os.makedirs('../data', exist_ok=True)
data.to_csv('../data/clean_data.csv', index=False)   # features + Price (dollars)
print('saved')

In [ ]:
# check that the saved file loads and gives the same answer
with open('../house_price_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
print(f"${np.exp(loaded_model.predict(new_house))[0]:,.0f}")